# Setup

## Importando Bibliotecas

In [8]:
import pandas as pd
import duckdb
import numpy as np
import pickle
import os

from duckdb.duckdb import DuckDBPyConnection
from pandas import DataFrame

import gc
import psutil

final_conn = duckdb.connect('final_db')

## Funções Auxiliares

In [9]:
def sql(
    query:str,
    conn:DuckDBPyConnection=final_conn
) -> DataFrame:
    """
    Executa uma consulta SQL em uma conexão de banco duckDb e retorna o resultado em um dataframe Pandas.

    Parâmetros
    ----------

    query (str)
        Query SQL a ser realizada no banco
    
    conn (DuckDBPyConnection)
        Conexão com banco DB em que a consulta será realizada
    
    Returns
    ----------

    DataFrame: Dataframe do Pandas com o resultado da consulta SQL realizada

    """
    
    df = conn.execute(query).fetch_df()
    return df

def check_memory():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / (1024 ** 2)  # Converte bytes para MB

## Importando bases

In [10]:
df_abt_base = sql("SELECT * FROM abt_base WHERE percentil_temporal > 0.7")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Escoragem

## Modelos 1

In [11]:
# Carregando Modelos
list_versions_1 = ['1_1', '1_2', '1_3', '1_4']
modelos_1 = {}

print("Importando modelos ...")
for version in list_versions_1:
    caminho = f'modelos_finais/modelo_{version}.pkl'
    try:
        with open(caminho, 'rb') as f:
            modelos_1[version] = pickle.load(f)
        
        print(f'✅ Versão {version} carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho} não encontrado.')
    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_1 = df_abt_base.copy()

for version, model in modelos_1.items():
    try:
        if hasattr(model, 'feature_names_in_'):
            features = model.feature_names_in_
            df_big_model_1[f'vl_output_model_{version}'] = model.predict(df_big_model_1[features])
        else:
            df_big_model_1[f'vl_output_model_{version}'] = model.predict(df_big_model_1)
            
        print(f'✅ Coluna "vl_output_model_{version}" adicionada.')
        
    except Exception as e:
        print(f'❌ Erro ao processar versão {version}: {e}')

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_1 AS (SELECT * FROM df_big_model_1)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_1
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 1_1 carregada com sucesso!
✅ Versão 1_2 carregada com sucesso!
✅ Versão 1_3 carregada com sucesso!
✅ Versão 1_4 carregada com sucesso!

Realizando a escoragem ...
✅ Coluna "vl_output_model_1_1" adicionada.
✅ Coluna "vl_output_model_1_2" adicionada.
✅ Coluna "vl_output_model_1_3" adicionada.
✅ Coluna "vl_output_model_1_4" adicionada.

Escrevendo a tabela big ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 15595.67 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 11441.42 MB
📉 Economia real: 4154.25 MB


## Modelos 2

In [12]:
# Carregando Modelos
list_versions_2 = ['2_1', '2_2', '2_3', '2_4']
modelos_2 = {}

print("Importando modelos ...")
for version in list_versions_2:
    caminho = f'modelos_finais/modelo_{version}.pkl'
    try:
        with open(caminho, 'rb') as f:
            modelos_2[version] = pickle.load(f)
        
        print(f'✅ Versão {version} carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho} não encontrado.')
    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_2 = df_abt_base.copy()

for version, model in modelos_2.items():
    try:
        if hasattr(model, 'feature_names_in_'):
            features = model.feature_names_in_
            df_big_model_2[f'vl_output_model_{version}'] = model.predict(df_big_model_2[features])
        else:
            df_big_model_2[f'vl_output_model_{version}'] = model.predict(df_big_model_2)
            
        print(f'✅ Coluna "vl_output_model_{version}" adicionada.')
        
    except Exception as e:
        print(f'❌ Erro ao processar versão {version}: {e}')

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_2 AS (SELECT * FROM df_big_model_2)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_2
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 2_1 carregada com sucesso!
✅ Versão 2_2 carregada com sucesso!
✅ Versão 2_3 carregada com sucesso!
✅ Versão 2_4 carregada com sucesso!

Realizando a escoragem ...
✅ Coluna "vl_output_model_2_1" adicionada.
✅ Coluna "vl_output_model_2_2" adicionada.
✅ Coluna "vl_output_model_2_3" adicionada.
✅ Coluna "vl_output_model_2_4" adicionada.

Escrevendo a tabela big ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 18086.09 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 13931.84 MB
📉 Economia real: 4154.26 MB


## Modelos 3

In [13]:
# Carregando Modelos
list_versions_3 = ['3_1', '3_2', '3_3', '3_4', '3_5']
modelos_3_especializado,modelos_3_geral = {},{}
dict_threshold_especializado = {
    '3_1':0.5,
    '3_2':0.75,
    '3_3':0.9,
    '3_4':0.4,
    '3_5':0.3
}

print("Importando modelos ...")
for version in list_versions_3:
    threshold = dict_threshold_especializado[version]
    caminho_especializado = f'modelos_finais/modelo_{version}_especializado.pkl'
    caminho_geral = f'modelos_finais/modelo_{version}_geral.pkl'
    try:
        with open(caminho, 'rb') as f:
            modelos_3_especializado[version] = pickle.load(f)
        
        print(f'✅ Versão {version} especializada carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_especializado} não encontrado.')
    try:
        with open(caminho_geral, 'rb') as f:
            modelos_3_geral[version] = pickle.load(f)
        
        print(f'✅ Versão {version} geral carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_geral} não encontrado.')

    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_3 = df_abt_base.copy()

for version in list_versions_3:
    modelo_especializado = modelos_3_especializado[version]
    modelo_geral = modelos_3_geral[version]

    try:
        features = model.feature_names_in_
        df_big_model_3[f'vl_output_model_especializado_{version}'] = modelo_especializado.predict(df_big_model_3[features])
        df_big_model_3[f'vl_output_model_geral_{version}'] = modelo_geral.predict(df_big_model_3[features])
        df_big_model_3[f'vl_output_model_{version}'] = np.where(
            df_big_model_3['vl_prioridade_vizinha_1']>=threshold,
            df_big_model_3[f'vl_output_model_especializado_{version}'],
            df_big_model_3[f'vl_output_model_geral_{version}']
            )
        df_big_model_3[f'vl_threshold_especializado_{version}'] = threshold

        print(f'✅ Colunas "vl_output_model_especializado_{version}", "vl_output_model_geral_{version}", "vl_threshold_especializado_{version}" e "vl_output_model_{version}" adicionadas.')
        
    except Exception as e:
        print(f'❌ Erro ao processar versão {version}: {e}')

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_3 AS (SELECT * FROM df_big_model_3)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_3
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 3_1 especializada carregada com sucesso!
✅ Versão 3_1 geral carregada com sucesso!
✅ Versão 3_2 especializada carregada com sucesso!
✅ Versão 3_2 geral carregada com sucesso!
✅ Versão 3_3 especializada carregada com sucesso!
✅ Versão 3_3 geral carregada com sucesso!
✅ Versão 3_4 especializada carregada com sucesso!
✅ Versão 3_4 geral carregada com sucesso!
✅ Versão 3_5 especializada carregada com sucesso!
✅ Versão 3_5 geral carregada com sucesso!

Realizando a escoragem ...
✅ Colunas "vl_output_model_especializado_3_1", "vl_output_model_geral_3_1", "vl_threshold_especializado_3_1" e "vl_output_model_3_1" adicionadas.
✅ Colunas "vl_output_model_especializado_3_2", "vl_output_model_geral_3_2", "vl_threshold_especializado_3_2" e "vl_output_model_3_2" adicionadas.
✅ Colunas "vl_output_model_especializado_3_3", "vl_output_model_geral_3_3", "vl_threshold_especializado_3_3" e "vl_output_model_3_3" adicionadas.
✅ Colunas "vl_output_model_especializado_3_4", "vl_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 19465.34 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 14874.83 MB
📉 Economia real: 4590.51 MB


## Modelos 4

In [14]:
# Carregando Modelos
list_versions_4 = ['4_1', '4_2', '4_3', '4_4', '4_5']
modelos_4_regressao,modelos_4_classificacao = {},{}
dict_classes = {
    '4_1':{
        0:(0,1),
        1:(1,np.inf)
        },
    '4_2':{
        0:(0,1),
        1:(1,51),
        2:(51,np.inf)
        },
    '4_3':{
        0:(0,1),
        1:(1,23),
        2:(23,51),
        3:(51,np.inf)
        },
    '4_4':{
        0:(0,1),
        1:(1,7),
        2:(7,51),
        3:(51,np.inf)
        },
    '4_5':{
        0:(0,1),
        1:(1,7),
        2:(7,np.inf)
        }
}

print("Importando modelos ...")
for version in list_versions_4:
    modelos_4_regressao[version] = {}
    caminho_classificacao = f'modelos_finais/modelo_{version}_classificacao.pkl'
    try:
        with open(caminho_classificacao, 'rb') as f:
            modelos_4_classificacao[version] = pickle.load(f)
        
        print(f'✅ Versão {version} classificação carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_classificacao} não encontrado.')
    
    classes = dict_classes[version].keys()
    for classe in classes:
        caminho_regressao_classe = f'modelos_finais/modelo_{version}_regressao_classe_{classe}.pkl'
        try:
            with open(caminho_regressao_classe, 'rb') as f:
                modelos_4_regressao[version][classe] = pickle.load(f)
            
            print(f'✅ Versão {version} regressão classe {classe} carregada com sucesso!')

        except FileNotFoundError:
            print(f'❌ Erro: Arquivo {caminho_regressao_classe} não encontrado.')

        except Exception as e:
            print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_4 = df_abt_base.copy()

for version in list_versions_4:
    modelo_classificacao = modelos_4_classificacao[version]
    try:
        features = modelo_classificacao.feature_names_in_
        df_big_model_4[f'vl_output_model_classificacao_{version}'] = modelo_classificacao.predict(df_big_model_4[features])
        df_big_model_4[f'vl_output_model_classificacao_{version}'] = df_big_model_4[f'vl_output_model_classificacao_{version}'].fillna(-9999).astype(int)
    
    except Exception as e:
        print(f'❌ Erro ao processar versão {version} de Classificação: {e}')

    classes = dict_classes[version]
    df_big_model_4[f'dict_threshold_classes_{version}'] = classes
    df_big_model_4[f'dict_threshold_classes_{version}'] = df_big_model_4[f'dict_threshold_classes_{version}'].astype(str)
    for classe in classes:
        try:
            modelo_regressao = modelos_4_regressao[version][classe]
            features = modelo_regressao.feature_names_in_
            df_big_model_4[f'vl_output_model_regressao_{version}_classe_{classe}'] = modelo_regressao.predict(df_big_model_4[features])
        except Exception as e:
            print(f'❌ Erro ao processar versão {version} de Regressão: {e}')

for version in list_versions_4:
    classe_predita = df_big_model_4[f'vl_output_model_classificacao_{version}']
    classes = dict_classes[version].keys()

    df_big_model_4[f'vl_output_model_{version}'] = np.nan

    for classe in classes:
        mask = classe_predita == classe
        df_big_model_4.loc[mask, f'vl_output_model_{version}'] = \
            df_big_model_4.loc[mask, f'vl_output_model_regressao_{version}_classe_{classe}']

    print(
        f'✅ Colunas "vl_output_model_classificacao_{version}", '
        f'"dict_threshold_classes_{version}", '
        f'"vl_output_model_regressao_{version}_classe_*" e '
        f'"vl_output_model_{version}" adicionadas.'
    )

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_4 AS (SELECT * FROM df_big_model_4)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_4
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 4_1 classificação carregada com sucesso!
✅ Versão 4_1 regressão classe 0 carregada com sucesso!
✅ Versão 4_1 regressão classe 1 carregada com sucesso!
✅ Versão 4_2 classificação carregada com sucesso!
✅ Versão 4_2 regressão classe 0 carregada com sucesso!
✅ Versão 4_2 regressão classe 1 carregada com sucesso!
✅ Versão 4_2 regressão classe 2 carregada com sucesso!
✅ Versão 4_3 classificação carregada com sucesso!
✅ Versão 4_3 regressão classe 0 carregada com sucesso!
✅ Versão 4_3 regressão classe 1 carregada com sucesso!
✅ Versão 4_3 regressão classe 2 carregada com sucesso!
✅ Versão 4_3 regressão classe 3 carregada com sucesso!
✅ Versão 4_4 classificação carregada com sucesso!
✅ Versão 4_4 regressão classe 0 carregada com sucesso!
✅ Versão 4_4 regressão classe 1 carregada com sucesso!
✅ Versão 4_4 regressão classe 2 carregada com sucesso!
✅ Versão 4_4 regressão classe 3 carregada com sucesso!
✅ Versão 4_5 classificação carregada com sucesso!
✅ Versão 4_5

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 15820.36 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 9255.29 MB
📉 Economia real: 6565.07 MB


## Modelos 5

In [15]:
# Carregando Modelos
list_versions_5 = ['5_1', '5_2', '5_3', '5_4', '5_5']
modelos_5_regressao,modelos_5_classificacao = {},{}
dict_classes = {
    '5_1':{
        0:(0,1),
        1:(1,np.inf)
        },
    '5_2':{
        0:(0,1),
        1:(1,51),
        2:(51,np.inf)
        },
    '5_3':{
        0:(0,1),
        1:(1,23),
        2:(23,51),
        3:(51,np.inf)
        },
    '5_4':{
        0:(0,1),
        1:(1,7),
        2:(7,51),
        3:(51,np.inf)
        },
    '5_5':{
        0:(0,1),
        1:(1,7),
        2:(7,np.inf)
        }
}

print("Importando modelos ...")
for version in list_versions_5:
    modelos_5_regressao[version] = {}
    caminho_classificacao = f'modelos_finais/modelo_{version}_classificacao.pkl'
    try:
        with open(caminho_classificacao, 'rb') as f:
            modelos_5_classificacao[version] = pickle.load(f)
        
        print(f'✅ Versão {version} classificação carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_classificacao} não encontrado.')
    
    classes = dict_classes[version].keys()
    for classe in classes:
        caminho_regressao_classe = f'modelos_finais/modelo_{version}_regressao_classe_{classe}.pkl'
        try:
            with open(caminho_regressao_classe, 'rb') as f:
                modelos_5_regressao[version][classe] = pickle.load(f)
            
            print(f'✅ Versão {version} regressão classe {classe} carregada com sucesso!')

        except FileNotFoundError:
            print(f'❌ Erro: Arquivo {caminho_regressao_classe} não encontrado.')

        except Exception as e:
            print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_5 = df_abt_base.copy()

for version in list_versions_5:
    modelo_classificacao = modelos_5_classificacao[version]
    classes = dict_classes[version]

    try:
        features = modelo_classificacao.feature_names_in_
        probas = modelo_classificacao.predict_proba(df_big_model_5[features])
        model_classes = modelo_classificacao.classes_

        for i, classe in enumerate(model_classes):
            col_proba = f'vl_output_model_classificacao_{version}_proba_classe_{classe}'
            df_big_model_5[col_proba] = probas[:, i]

    except Exception as e:
        print(f'❌ Erro ao processar versão {version} de Classificação: {e}')

    df_big_model_5[f'dict_threshold_classes_{version}'] = str(classes)

    for classe in classes:
        try:
            modelo_regressao = modelos_5_regressao[version][classe]
            features = modelo_regressao.feature_names_in_
            df_big_model_5[f'vl_output_model_regressao_{version}_classe_{classe}'] = modelo_regressao.predict(df_big_model_5[features])
        except Exception as e:
            print(f'❌ Erro ao processar versão {version} de Regressão: {e}')

for version in list_versions_5:
    classes = dict_classes[version]
    model_classes = modelos_5_classificacao[version].classes_

    df_big_model_5[f'vl_output_model_{version}'] = sum(
        df_big_model_5[f'vl_output_model_classificacao_{version}_proba_classe_{classe}'] *
        df_big_model_5[f'vl_output_model_regressao_{version}_classe_{classe}']
        for classe in model_classes
    )

    print(
        f'✅ Colunas "vl_output_model_classificacao_{version}_proba_classe_*", '
        f'"dict_threshold_classes_{version}", '
        f'"vl_output_model_regressao_{version}_classe_*" e '
        f'"vl_output_model_{version}" adicionadas.'
    )

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_5 AS (SELECT * FROM df_big_model_5)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_5
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 5_1 classificação carregada com sucesso!
✅ Versão 5_1 regressão classe 0 carregada com sucesso!
✅ Versão 5_1 regressão classe 1 carregada com sucesso!
✅ Versão 5_2 classificação carregada com sucesso!
✅ Versão 5_2 regressão classe 0 carregada com sucesso!
✅ Versão 5_2 regressão classe 1 carregada com sucesso!
✅ Versão 5_2 regressão classe 2 carregada com sucesso!
✅ Versão 5_3 classificação carregada com sucesso!
✅ Versão 5_3 regressão classe 0 carregada com sucesso!
✅ Versão 5_3 regressão classe 1 carregada com sucesso!
✅ Versão 5_3 regressão classe 2 carregada com sucesso!
✅ Versão 5_3 regressão classe 3 carregada com sucesso!
✅ Versão 5_4 classificação carregada com sucesso!
✅ Versão 5_4 regressão classe 0 carregada com sucesso!
✅ Versão 5_4 regressão classe 1 carregada com sucesso!
✅ Versão 5_4 regressão classe 2 carregada com sucesso!
✅ Versão 5_4 regressão classe 3 carregada com sucesso!
✅ Versão 5_5 classificação carregada com sucesso!
✅ Versão 5_5

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 16497.87 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 11450.33 MB
📉 Economia real: 5047.54 MB


## Modelos 6

In [16]:
# Carregando Modelos
list_versions_6 = ['6_1', '6_2', '6_3', '6_4', '6_5']

dict_classes_6 = {
    '6_1': {
        0: (0, 1),
        1: (1, np.inf)
    },
    '6_2': {
        0: (0, 1),
        1: (1, 51),
        2: (51, np.inf)
    },
    '6_3': {
        0: (0, 1),
        1: (1, 23),
        2: (23, 51),
        3: (51, np.inf)
    },
    '6_4': {
        0: (0, 1),
        1: (1, 7),
        2: (7, 51),
        3: (51, np.inf)
    },
    '6_5': {
        0: (0, 1),
        1: (1, 7),
        2: (7, np.inf)
    }
}

THRESHOLD_ESPECIALIZADO = 0.5
COLUNA_PRIORIDADE_VIZINHA = 'vl_prioridade_vizinha_1'

modelos_6_classificacao = {}
modelos_6_regressao_geral = {}
modelos_6_regressao_especializado = {}

print("Importando modelos ...")
for version in list_versions_6:
    modelos_6_regressao_geral[version] = {}
    modelos_6_regressao_especializado[version] = {}

    caminho_classificacao = f'modelos_finais/modelo_{version}_classificacao.pkl'
    try:
        with open(caminho_classificacao, 'rb') as f:
            modelos_6_classificacao[version] = pickle.load(f)
        print(f'✅ Versão {version} classificação carregada com sucesso!')
    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_classificacao} não encontrado.')
    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version} classificação: {e}')

    for classe in dict_classes_6[version].keys():
        caminho_geral = f'modelos_finais/modelo_{version}_regressao_geral_classe_{classe}.pkl'
        try:
            with open(caminho_geral, 'rb') as f:
                modelos_6_regressao_geral[version][classe] = pickle.load(f)
            print(f'✅ Versão {version} regressão geral classe {classe} carregada com sucesso!')
        except FileNotFoundError:
            print(f'❌ Erro: Arquivo {caminho_geral} não encontrado.')
        except Exception as e:
            print(f'⚠️ Erro inesperado na versão {version} regressão geral classe {classe}: {e}')

        caminho_especializado = f'modelos_finais/modelo_{version}_regressao_especializado_classe_{classe}.pkl'
        try:
            with open(caminho_especializado, 'rb') as f:
                modelos_6_regressao_especializado[version][classe] = pickle.load(f)
            print(f'✅ Versão {version} regressão especializado classe {classe} carregada com sucesso!')
        except FileNotFoundError:
            print(f'❌ Erro: Arquivo {caminho_especializado} não encontrado.')
        except Exception as e:
            print(f'⚠️ Erro inesperado na versão {version} regressão especializado classe {classe}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_6 = df_abt_base.copy()

df_big_model_6['flag_especializado'] = (
    df_big_model_6[COLUNA_PRIORIDADE_VIZINHA] >= THRESHOLD_ESPECIALIZADO
).astype(int)

for version in list_versions_6:
    modelo_classificacao = modelos_6_classificacao[version]
    classes = dict_classes_6[version]

    try:
        features_classificacao = modelo_classificacao.feature_names_in_
        classe_predita = modelo_classificacao.predict(df_big_model_6[features_classificacao])
        df_big_model_6[f'vl_output_model_classificacao_{version}_classe_predita'] = classe_predita
    except Exception as e:
        print(f'❌ Erro ao processar versão {version} de Classificação: {e}')
        continue

    df_big_model_6[f'dict_threshold_classes_{version}'] = str(classes)

    col_output = f'vl_output_model_{version}'
    df_big_model_6[col_output] = np.nan

    for classe in classes.keys():
        mask_classe = df_big_model_6[f'vl_output_model_classificacao_{version}_classe_predita'] == classe
        mask_especializado = df_big_model_6['flag_especializado'] == 1
        mask_usar_especializado = mask_classe & mask_especializado & (classe in modelos_6_regressao_especializado[version])
        mask_usar_geral = mask_classe & ~mask_usar_especializado

        if mask_usar_especializado.any():
            try:
                modelo_esp = modelos_6_regressao_especializado[version][classe]
                features_esp = modelo_esp.feature_names_in_
                df_big_model_6.loc[mask_usar_especializado, col_output] = modelo_esp.predict(
                    df_big_model_6.loc[mask_usar_especializado, features_esp]
                )
            except Exception as e:
                print(f'❌ Erro ao processar versão {version} regressão especializado classe {classe}: {e}')

        if mask_usar_geral.any():
            try:
                modelo_ger = modelos_6_regressao_geral[version][classe]
                features_ger = modelo_ger.feature_names_in_
                df_big_model_6.loc[mask_usar_geral, col_output] = modelo_ger.predict(
                    df_big_model_6.loc[mask_usar_geral, features_ger]
                )
            except Exception as e:
                print(f'❌ Erro ao processar versão {version} regressão geral classe {classe}: {e}')

    print(
        f'✅ Colunas "vl_output_model_classificacao_{version}_classe_predita", '
        f'"dict_threshold_classes_{version}" e '
        f'"vl_output_model_{version}" adicionadas.'
    )

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_6 AS (SELECT * FROM df_big_model_6)")
print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_6
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 6_1 classificação carregada com sucesso!
✅ Versão 6_1 regressão geral classe 0 carregada com sucesso!
✅ Versão 6_1 regressão especializado classe 0 carregada com sucesso!
✅ Versão 6_1 regressão geral classe 1 carregada com sucesso!
✅ Versão 6_1 regressão especializado classe 1 carregada com sucesso!
✅ Versão 6_2 classificação carregada com sucesso!
✅ Versão 6_2 regressão geral classe 0 carregada com sucesso!
✅ Versão 6_2 regressão especializado classe 0 carregada com sucesso!
✅ Versão 6_2 regressão geral classe 1 carregada com sucesso!
✅ Versão 6_2 regressão especializado classe 1 carregada com sucesso!
✅ Versão 6_2 regressão geral classe 2 carregada com sucesso!
✅ Versão 6_2 regressão especializado classe 2 carregada com sucesso!
✅ Versão 6_3 classificação carregada com sucesso!
✅ Versão 6_3 regressão geral classe 0 carregada com sucesso!
✅ Versão 6_3 regressão especializado classe 0 carregada com sucesso!
✅ Versão 6_3 regressão geral classe 1 carregada

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 14346.26 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 9651.95 MB
📉 Economia real: 4694.32 MB


## Modelos 7

In [17]:
# Carregando Modelos
list_versions_7 = ['7_1', '7_2', '7_3', '7_4', '7_5']

dict_classes_7 = {
    '7_1': {
        0: (0, 1),
        1: (1, np.inf)
    },
    '7_2': {
        0: (0, 1),
        1: (1, 51),
        2: (51, np.inf)
    },
    '7_3': {
        0: (0, 1),
        1: (1, 23),
        2: (23, 51),
        3: (51, np.inf)
    },
    '7_4': {
        0: (0, 1),
        1: (1, 7),
        2: (7, 51),
        3: (51, np.inf)
    },
    '7_5': {
        0: (0, 1),
        1: (1, 7),
        2: (7, np.inf)
    }
}

THRESHOLD_ESPECIALIZADO = 0.5
COLUNA_PRIORIDADE_VIZINHA = 'vl_prioridade_vizinha_1'

modelos_7_classificacao = {}
modelos_7_regressao_geral = {}
modelos_7_regressao_especializado = {}

print("Importando modelos ...")
for version in list_versions_7:
    modelos_7_regressao_geral[version] = {}
    modelos_7_regressao_especializado[version] = {}

    caminho_classificacao = f'modelos_finais/modelo_{version}_classificacao.pkl'
    try:
        with open(caminho_classificacao, 'rb') as f:
            modelos_7_classificacao[version] = pickle.load(f)
        print(f'✅ Versão {version} classificação carregada com sucesso!')
    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_classificacao} não encontrado.')
    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version} classificação: {e}')

    for classe in dict_classes_7[version].keys():
        caminho_geral = f'modelos_finais/modelo_{version}_regressao_geral_classe_{classe}.pkl'
        try:
            with open(caminho_geral, 'rb') as f:
                modelos_7_regressao_geral[version][classe] = pickle.load(f)
            print(f'✅ Versão {version} regressão geral classe {classe} carregada com sucesso!')
        except FileNotFoundError:
            print(f'❌ Erro: Arquivo {caminho_geral} não encontrado.')
        except Exception as e:
            print(f'⚠️ Erro inesperado na versão {version} regressão geral classe {classe}: {e}')

        caminho_especializado = f'modelos_finais/modelo_{version}_regressao_especializado_classe_{classe}.pkl'
        try:
            with open(caminho_especializado, 'rb') as f:
                modelos_7_regressao_especializado[version][classe] = pickle.load(f)
            print(f'✅ Versão {version} regressão especializado classe {classe} carregada com sucesso!')
        except FileNotFoundError:
            print(f'❌ Erro: Arquivo {caminho_especializado} não encontrado.')
        except Exception as e:
            print(f'⚠️ Erro inesperado na versão {version} regressão especializado classe {classe}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_7 = df_abt_base.copy()

df_big_model_7['flag_especializado'] = (
    df_big_model_7[COLUNA_PRIORIDADE_VIZINHA] >= THRESHOLD_ESPECIALIZADO
).astype(int)

for version in list_versions_7:
    modelo_classificacao = modelos_7_classificacao[version]
    classes = dict_classes_7[version]
    model_classes = modelo_classificacao.classes_

    try:
        features_classificacao = modelo_classificacao.feature_names_in_
        classe_predita = modelo_classificacao.predict(df_big_model_7[features_classificacao])
        probas = modelo_classificacao.predict_proba(df_big_model_7[features_classificacao])

        df_big_model_7[f'vl_output_model_classificacao_{version}_classe_predita'] = classe_predita

        for i, classe in enumerate(model_classes):
            df_big_model_7[f'vl_output_model_classificacao_{version}_proba_classe_{classe}'] = probas[:, i]

    except Exception as e:
        print(f'❌ Erro ao processar versão {version} de Classificação: {e}')
        continue

    df_big_model_7[f'dict_threshold_classes_{version}'] = str(classes)

    for classe in classes.keys():
        for tipo, dict_modelos in [('geral', modelos_7_regressao_geral), ('especializado', modelos_7_regressao_especializado)]:
            if classe not in dict_modelos[version]:
                continue
            try:
                modelo_reg = dict_modelos[version][classe]
                features_reg = modelo_reg.feature_names_in_
                df_big_model_7[f'vl_output_model_regressao_{version}_{tipo}_classe_{classe}'] = modelo_reg.predict(
                    df_big_model_7[features_reg]
                )
            except Exception as e:
                print(f'❌ Erro ao processar versão {version} regressão {tipo} classe {classe}: {e}')

    mask_especializado = df_big_model_7['flag_especializado'] == 1
    col_output = f'vl_output_model_{version}'
    df_big_model_7[col_output] = np.nan

    # Classe 0: predição direta, sem ponderação
    mask_classe_0 = df_big_model_7[f'vl_output_model_classificacao_{version}_classe_predita'] == 0

    mask_c0_esp = mask_classe_0 & mask_especializado & (0 in modelos_7_regressao_especializado[version])
    mask_c0_ger = mask_classe_0 & ~mask_c0_esp

    if mask_c0_esp.any():
        col_pred_esp = f'vl_output_model_regressao_{version}_especializado_classe_0'
        if col_pred_esp in df_big_model_7.columns:
            df_big_model_7.loc[mask_c0_esp, col_output] = df_big_model_7.loc[mask_c0_esp, col_pred_esp]

    if mask_c0_ger.any():
        col_pred_ger = f'vl_output_model_regressao_{version}_geral_classe_0'
        if col_pred_ger in df_big_model_7.columns:
            df_big_model_7.loc[mask_c0_ger, col_output] = df_big_model_7.loc[mask_c0_ger, col_pred_ger]

    # Outras classes: soma ponderada Σ proba_classe × pred_regressao_classe
    mask_outras = df_big_model_7[f'vl_output_model_classificacao_{version}_classe_predita'] != 0

    for tipo, dict_modelos, mask_tipo in [
        ('especializado', modelos_7_regressao_especializado, mask_outras & mask_especializado),
        ('geral',         modelos_7_regressao_geral,         mask_outras & ~mask_especializado),
    ]:
        if not mask_tipo.any():
            continue

        classes_nao_zero = [c for c in model_classes if c != 0]
        cols_ponderadas = []

        for classe in classes_nao_zero:
            col_pred = f'vl_output_model_regressao_{version}_{tipo}_classe_{classe}'
            col_proba = f'vl_output_model_classificacao_{version}_proba_classe_{classe}'

            if col_pred not in df_big_model_7.columns or col_proba not in df_big_model_7.columns:
                continue

            cols_ponderadas.append(
                df_big_model_7.loc[mask_tipo, col_pred] * df_big_model_7.loc[mask_tipo, col_proba]
            )

        if cols_ponderadas:
            df_big_model_7.loc[mask_tipo, col_output] = sum(cols_ponderadas)

    print(
        f'✅ Colunas "vl_output_model_classificacao_{version}_classe_predita", '
        f'"vl_output_model_classificacao_{version}_proba_classe_*", '
        f'"dict_threshold_classes_{version}", '
        f'"vl_output_model_regressao_{version}_geral_classe_*", '
        f'"vl_output_model_regressao_{version}_especializado_classe_*" e '
        f'"vl_output_model_{version}" adicionadas.'
    )

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_7 AS (SELECT * FROM df_big_model_7)")
print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_7
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 7_1 classificação carregada com sucesso!
✅ Versão 7_1 regressão geral classe 0 carregada com sucesso!
✅ Versão 7_1 regressão especializado classe 0 carregada com sucesso!
✅ Versão 7_1 regressão geral classe 1 carregada com sucesso!
✅ Versão 7_1 regressão especializado classe 1 carregada com sucesso!
✅ Versão 7_2 classificação carregada com sucesso!
✅ Versão 7_2 regressão geral classe 0 carregada com sucesso!
✅ Versão 7_2 regressão especializado classe 0 carregada com sucesso!
✅ Versão 7_2 regressão geral classe 1 carregada com sucesso!
✅ Versão 7_2 regressão especializado classe 1 carregada com sucesso!
✅ Versão 7_2 regressão geral classe 2 carregada com sucesso!
✅ Versão 7_2 regressão especializado classe 2 carregada com sucesso!
✅ Versão 7_3 classificação carregada com sucesso!
✅ Versão 7_3 regressão geral classe 0 carregada com sucesso!
✅ Versão 7_3 regressão especializado classe 0 carregada com sucesso!
✅ Versão 7_3 regressão geral classe 1 carregada

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 18084.05 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 12285.90 MB
📉 Economia real: 5798.15 MB


## Organização

In [18]:
sql(f"""
SHOW TABLES
""")

,name
0,abt_base
1,big_model_1
2,big_model_2
3,big_model_3
4,big_model_4
5,big_model_5
6,big_model_6
7,big_model_7
8,big_union
9,tb_metricas_classificacao


In [19]:
for model_index in range(1,8):
    print(model_index)

    print(','.join([c for c in sql(
    f"""
    SELECT 
        *
    FROM big_model_{model_index}
    LIMIT 100
    """).columns if 'model' in c]))

1
vl_output_model_1_1,vl_output_model_1_2,vl_output_model_1_3,vl_output_model_1_4
2
vl_output_model_2_1,vl_output_model_2_2,vl_output_model_2_3,vl_output_model_2_4
3
vl_output_model_especializado_3_1,vl_output_model_geral_3_1,vl_output_model_3_1,vl_output_model_especializado_3_2,vl_output_model_geral_3_2,vl_output_model_3_2,vl_output_model_especializado_3_3,vl_output_model_geral_3_3,vl_output_model_3_3,vl_output_model_especializado_3_4,vl_output_model_geral_3_4,vl_output_model_3_4,vl_output_model_especializado_3_5,vl_output_model_geral_3_5,vl_output_model_3_5
4
vl_output_model_classificacao_4_1,vl_output_model_regressao_4_1_classe_0,vl_output_model_regressao_4_1_classe_1,vl_output_model_classificacao_4_2,vl_output_model_regressao_4_2_classe_0,vl_output_model_regressao_4_2_classe_1,vl_output_model_regressao_4_2_classe_2,vl_output_model_classificacao_4_3,vl_output_model_regressao_4_3_classe_0,vl_output_model_regressao_4_3_classe_1,vl_output_model_regressao_4_3_classe_2,vl_output_model_re

In [20]:
list(sql("select * from big_model_1 limit 1").columns)

['id_estacao',
 'latitude',
 'longitude',
 'vl_declividade',
 'vl_altitude',
 'vl_distancia_oceano',
 'vl_aspecto_relevo',
 'dt_medicao',
 'vl_precipitacao',
 'vl_temperatura_maxima',
 'vl_temperatura_media',
 'vl_temperatura_minima',
 'vl_umidade_relativa_maxima',
 'vl_umidade_relativa_media',
 'vl_umidade_relativa_minima',
 'vl_velocidade_vento_2m_maxima',
 'vl_velocidade_vento_2m_media',
 'vl_velocidade_vento_10m_media',
 'vl_precipitacao_chirps',
 'vl_precipitacao_cpc',
 'vl_temperatura_maxima_cpc',
 'vl_temperatura_minima_cpc',
 'vl_precipitacao_gpm_final_run',
 'vl_precipitacao_gpm_late_run',
 'vl_precipitacao_power',
 'vl_temperatura_maxima_2m_K_power',
 'vl_temperatura_media_2m_K_power',
 'vl_temperatura_minima_2m_K_power',
 'vl_umidade_relativa_2m_power',
 'vl_pressao_nivel_superficie_power',
 'vl_irradiancia_allsky_power',
 'vl_direcao_vento_10m_power',
 'vl_direcao_vento_2m_power',
 'vl_temperatura_orvalho_2m_K_power',
 'vl_vento_10m_power',
 'vl_vento_medio_2m_power',
 'vl_

In [21]:
sql(
"""
CREATE OR REPLACE TABLE big_union AS (
SELECT
    -- Keys
    b1.id_estacao,
    b1.dt_medicao,

    -- Variáveis explicativas selecionadas
    b1.latitude,
    b1.longitude,
    b1.vl_declividade,
    b1.vl_altitude,
    b1.vl_distancia_oceano,
    b1.vl_aspecto_relevo,
    b1.vl_precipitacao,
    b1.vl_prioridade_vizinha_1,

    -- Modelo 1
    b1.vl_output_model_1_1,
    b1.vl_output_model_1_2,
    b1.vl_output_model_1_3,
    b1.vl_output_model_1_4,

    -- Modelo 2
    b2.vl_output_model_2_1,
    b2.vl_output_model_2_2,
    b2.vl_output_model_2_3,
    b2.vl_output_model_2_4,

    -- Modelo 3
    b3.vl_output_model_especializado_3_1,
    b3.vl_output_model_geral_3_1,
    b3.vl_output_model_3_1,
    b3.vl_output_model_especializado_3_2,
    b3.vl_output_model_geral_3_2,
    b3.vl_output_model_3_2,
    b3.vl_output_model_especializado_3_3,
    b3.vl_output_model_geral_3_3,
    b3.vl_output_model_3_3,
    b3.vl_output_model_especializado_3_4,
    b3.vl_output_model_geral_3_4,
    b3.vl_output_model_3_4,
    b3.vl_output_model_especializado_3_5,
    b3.vl_output_model_geral_3_5,
    b3.vl_output_model_3_5,

    -- Modelo 4
    b4.vl_output_model_classificacao_4_1,
    b4.vl_output_model_regressao_4_1_classe_0,
    b4.vl_output_model_regressao_4_1_classe_1,
    b4.vl_output_model_classificacao_4_2,
    b4.vl_output_model_regressao_4_2_classe_0,
    b4.vl_output_model_regressao_4_2_classe_1,
    b4.vl_output_model_regressao_4_2_classe_2,
    b4.vl_output_model_classificacao_4_3,
    b4.vl_output_model_regressao_4_3_classe_0,
    b4.vl_output_model_regressao_4_3_classe_1,
    b4.vl_output_model_regressao_4_3_classe_2,
    b4.vl_output_model_regressao_4_3_classe_3,
    b4.vl_output_model_classificacao_4_4,
    b4.vl_output_model_regressao_4_4_classe_0,
    b4.vl_output_model_regressao_4_4_classe_1,
    b4.vl_output_model_regressao_4_4_classe_2,
    b4.vl_output_model_regressao_4_4_classe_3,
    b4.vl_output_model_classificacao_4_5,
    b4.vl_output_model_regressao_4_5_classe_0,
    b4.vl_output_model_regressao_4_5_classe_1,
    b4.vl_output_model_regressao_4_5_classe_2,
    b4.vl_output_model_4_1,
    b4.vl_output_model_4_2,
    b4.vl_output_model_4_3,
    b4.vl_output_model_4_4,
    b4.vl_output_model_4_5,

    -- Modelo 5
    b5.vl_output_model_classificacao_5_1_proba_classe_0,
    b5.vl_output_model_classificacao_5_1_proba_classe_1,
    b5.vl_output_model_regressao_5_1_classe_0,
    b5.vl_output_model_regressao_5_1_classe_1,
    b5.vl_output_model_classificacao_5_2_proba_classe_0,
    b5.vl_output_model_classificacao_5_2_proba_classe_1,
    b5.vl_output_model_classificacao_5_2_proba_classe_2,
    b5.vl_output_model_regressao_5_2_classe_0,
    b5.vl_output_model_regressao_5_2_classe_1,
    b5.vl_output_model_regressao_5_2_classe_2,
    b5.vl_output_model_classificacao_5_3_proba_classe_0,
    b5.vl_output_model_classificacao_5_3_proba_classe_1,
    b5.vl_output_model_classificacao_5_3_proba_classe_2,
    b5.vl_output_model_classificacao_5_3_proba_classe_3,
    b5.vl_output_model_regressao_5_3_classe_0,
    b5.vl_output_model_regressao_5_3_classe_1,
    b5.vl_output_model_regressao_5_3_classe_2,
    b5.vl_output_model_regressao_5_3_classe_3,
    b5.vl_output_model_classificacao_5_4_proba_classe_0,
    b5.vl_output_model_classificacao_5_4_proba_classe_1,
    b5.vl_output_model_classificacao_5_4_proba_classe_2,
    b5.vl_output_model_classificacao_5_4_proba_classe_3,
    b5.vl_output_model_regressao_5_4_classe_0,
    b5.vl_output_model_regressao_5_4_classe_1,
    b5.vl_output_model_regressao_5_4_classe_2,
    b5.vl_output_model_regressao_5_4_classe_3,
    b5.vl_output_model_classificacao_5_5_proba_classe_0,
    b5.vl_output_model_classificacao_5_5_proba_classe_1,
    b5.vl_output_model_classificacao_5_5_proba_classe_2,
    b5.vl_output_model_regressao_5_5_classe_0,
    b5.vl_output_model_regressao_5_5_classe_1,
    b5.vl_output_model_regressao_5_5_classe_2,
    b5.vl_output_model_5_1,
    b5.vl_output_model_5_2,
    b5.vl_output_model_5_3,
    b5.vl_output_model_5_4,
    b5.vl_output_model_5_5,

    -- Modelo 6
    b6.vl_output_model_classificacao_6_1_classe_predita,
    b6.vl_output_model_6_1,
    b6.vl_output_model_classificacao_6_2_classe_predita,
    b6.vl_output_model_6_2,
    b6.vl_output_model_classificacao_6_3_classe_predita,
    b6.vl_output_model_6_3,
    b6.vl_output_model_classificacao_6_4_classe_predita,
    b6.vl_output_model_6_4,
    b6.vl_output_model_classificacao_6_5_classe_predita,
    b6.vl_output_model_6_5,

    -- Modelo 7
    b7.vl_output_model_classificacao_7_1_classe_predita,
    b7.vl_output_model_classificacao_7_1_proba_classe_0,
    b7.vl_output_model_classificacao_7_1_proba_classe_1,
    b7.vl_output_model_regressao_7_1_geral_classe_0,
    b7.vl_output_model_regressao_7_1_especializado_classe_0,
    b7.vl_output_model_regressao_7_1_geral_classe_1,
    b7.vl_output_model_regressao_7_1_especializado_classe_1,
    b7.vl_output_model_7_1,
    b7.vl_output_model_classificacao_7_2_classe_predita,
    b7.vl_output_model_classificacao_7_2_proba_classe_0,
    b7.vl_output_model_classificacao_7_2_proba_classe_1,
    b7.vl_output_model_classificacao_7_2_proba_classe_2,
    b7.vl_output_model_regressao_7_2_geral_classe_0,
    b7.vl_output_model_regressao_7_2_especializado_classe_0,
    b7.vl_output_model_regressao_7_2_geral_classe_1,
    b7.vl_output_model_regressao_7_2_especializado_classe_1,
    b7.vl_output_model_regressao_7_2_geral_classe_2,
    b7.vl_output_model_regressao_7_2_especializado_classe_2,
    b7.vl_output_model_7_2,
    b7.vl_output_model_classificacao_7_3_classe_predita,
    b7.vl_output_model_classificacao_7_3_proba_classe_0,
    b7.vl_output_model_classificacao_7_3_proba_classe_1,
    b7.vl_output_model_classificacao_7_3_proba_classe_2,
    b7.vl_output_model_classificacao_7_3_proba_classe_3,
    b7.vl_output_model_regressao_7_3_geral_classe_0,
    b7.vl_output_model_regressao_7_3_especializado_classe_0,
    b7.vl_output_model_regressao_7_3_geral_classe_1,
    b7.vl_output_model_regressao_7_3_especializado_classe_1,
    b7.vl_output_model_regressao_7_3_geral_classe_2,
    b7.vl_output_model_regressao_7_3_especializado_classe_2,
    b7.vl_output_model_regressao_7_3_geral_classe_3,
    b7.vl_output_model_regressao_7_3_especializado_classe_3,
    b7.vl_output_model_7_3,
    b7.vl_output_model_classificacao_7_4_classe_predita,
    b7.vl_output_model_classificacao_7_4_proba_classe_0,
    b7.vl_output_model_classificacao_7_4_proba_classe_1,
    b7.vl_output_model_classificacao_7_4_proba_classe_2,
    b7.vl_output_model_classificacao_7_4_proba_classe_3,
    b7.vl_output_model_regressao_7_4_geral_classe_0,
    b7.vl_output_model_regressao_7_4_especializado_classe_0,
    b7.vl_output_model_regressao_7_4_geral_classe_1,
    b7.vl_output_model_regressao_7_4_especializado_classe_1,
    b7.vl_output_model_regressao_7_4_geral_classe_2,
    b7.vl_output_model_regressao_7_4_especializado_classe_2,
    b7.vl_output_model_regressao_7_4_geral_classe_3,
    b7.vl_output_model_regressao_7_4_especializado_classe_3,
    b7.vl_output_model_7_4,
    b7.vl_output_model_classificacao_7_5_classe_predita,
    b7.vl_output_model_classificacao_7_5_proba_classe_0,
    b7.vl_output_model_classificacao_7_5_proba_classe_1,
    b7.vl_output_model_classificacao_7_5_proba_classe_2,
    b7.vl_output_model_regressao_7_5_geral_classe_0,
    b7.vl_output_model_regressao_7_5_especializado_classe_0,
    b7.vl_output_model_regressao_7_5_geral_classe_1,
    b7.vl_output_model_regressao_7_5_especializado_classe_1,
    b7.vl_output_model_regressao_7_5_geral_classe_2,
    b7.vl_output_model_regressao_7_5_especializado_classe_2,
    b7.vl_output_model_7_5

FROM      big_model_1 b1
LEFT JOIN big_model_2 b2 ON b1.id_estacao = b2.id_estacao AND b1.dt_medicao = b2.dt_medicao
LEFT JOIN big_model_3 b3 ON b1.id_estacao = b3.id_estacao AND b1.dt_medicao = b3.dt_medicao
LEFT JOIN big_model_4 b4 ON b1.id_estacao = b4.id_estacao AND b1.dt_medicao = b4.dt_medicao
LEFT JOIN big_model_5 b5 ON b1.id_estacao = b5.id_estacao AND b1.dt_medicao = b5.dt_medicao
LEFT JOIN big_model_6 b6 ON b1.id_estacao = b6.id_estacao AND b1.dt_medicao = b6.dt_medicao
LEFT JOIN big_model_7 b7 ON b1.id_estacao = b7.id_estacao AND b1.dt_medicao = b7.dt_medicao
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,5445084


# Cálculo de Métricas Gerais

## Setup

In [22]:
import pandas as pd
import duckdb
import numpy as np
import pickle
import os

from duckdb.duckdb import DuckDBPyConnection
from pandas import DataFrame

import gc
import psutil

final_conn = duckdb.connect('final_db')

In [23]:
def sql(
    query:str,
    conn:DuckDBPyConnection=final_conn
) -> DataFrame:
    """
    Executa uma consulta SQL em uma conexão de banco duckDb e retorna o resultado em um dataframe Pandas.

    Parâmetros
    ----------

    query (str)
        Query SQL a ser realizada no banco
    
    conn (DuckDBPyConnection)
        Conexão com banco DB em que a consulta será realizada
    
    Returns
    ----------

    DataFrame: Dataframe do Pandas com o resultado da consulta SQL realizada

    """
    
    df = conn.execute(query).fetch_df()
    return df

def check_memory():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / (1024 ** 2)  # Converte bytes para MB

## Importação da Base

In [24]:
df_big = sql(
"""
SELECT 
    id_estacao
    ,dt_medicao
    ,vl_precipitacao
    ,vl_output_model_1_1
    ,vl_output_model_1_2
    ,vl_output_model_1_3
    ,vl_output_model_1_4
    ,vl_output_model_2_1
    ,vl_output_model_2_2
    ,vl_output_model_2_3
    ,vl_output_model_2_4
    ,vl_output_model_3_1
    ,vl_output_model_3_2
    ,vl_output_model_3_3
    ,vl_output_model_3_4
    ,vl_output_model_3_5
    ,vl_output_model_classificacao_4_1
    ,vl_output_model_4_1
    ,vl_output_model_classificacao_4_2
    ,vl_output_model_4_2
    ,vl_output_model_classificacao_4_3
    ,vl_output_model_4_3
    ,vl_output_model_classificacao_4_4
    ,vl_output_model_4_4
    ,vl_output_model_classificacao_4_5
    ,vl_output_model_4_5
    ,vl_output_model_classificacao_5_1_proba_classe_0
    ,vl_output_model_classificacao_5_1_proba_classe_1
    ,vl_output_model_5_1
    ,vl_output_model_classificacao_5_2_proba_classe_0
    ,vl_output_model_classificacao_5_2_proba_classe_1
    ,vl_output_model_classificacao_5_2_proba_classe_2
    ,vl_output_model_5_2
    ,vl_output_model_classificacao_5_3_proba_classe_0
    ,vl_output_model_classificacao_5_3_proba_classe_1
    ,vl_output_model_classificacao_5_3_proba_classe_2
    ,vl_output_model_classificacao_5_3_proba_classe_3
    ,vl_output_model_5_3
    ,vl_output_model_classificacao_5_4_proba_classe_0
    ,vl_output_model_classificacao_5_4_proba_classe_1
    ,vl_output_model_classificacao_5_4_proba_classe_2
    ,vl_output_model_classificacao_5_4_proba_classe_3
    ,vl_output_model_5_4
    ,vl_output_model_classificacao_5_5_proba_classe_0
    ,vl_output_model_classificacao_5_5_proba_classe_1
    ,vl_output_model_classificacao_5_5_proba_classe_2
    ,vl_output_model_5_5
    ,vl_output_model_classificacao_6_1_classe_predita AS vl_output_model_classificacao_6_1
    ,vl_output_model_6_1
    ,vl_output_model_classificacao_6_2_classe_predita AS vl_output_model_classificacao_6_2
    ,vl_output_model_6_2
    ,vl_output_model_classificacao_6_3_classe_predita AS vl_output_model_classificacao_6_3
    ,vl_output_model_6_3
    ,vl_output_model_classificacao_6_4_classe_predita AS vl_output_model_classificacao_6_4
    ,vl_output_model_6_4
    ,vl_output_model_classificacao_6_5_classe_predita AS vl_output_model_classificacao_6_5
    ,vl_output_model_6_5
    ,vl_output_model_classificacao_7_1_proba_classe_0
    ,vl_output_model_classificacao_7_1_proba_classe_1
    ,vl_output_model_7_1
    ,vl_output_model_classificacao_7_2_proba_classe_0
    ,vl_output_model_classificacao_7_2_proba_classe_1
    ,vl_output_model_classificacao_7_2_proba_classe_2
    ,vl_output_model_7_2
    ,vl_output_model_classificacao_7_3_proba_classe_0
    ,vl_output_model_classificacao_7_3_proba_classe_1
    ,vl_output_model_classificacao_7_3_proba_classe_2
    ,vl_output_model_classificacao_7_3_proba_classe_3
    ,vl_output_model_7_3
    ,vl_output_model_classificacao_7_4_proba_classe_0
    ,vl_output_model_classificacao_7_4_proba_classe_1
    ,vl_output_model_classificacao_7_4_proba_classe_2
    ,vl_output_model_classificacao_7_4_proba_classe_3
    ,vl_output_model_7_4
    ,vl_output_model_classificacao_7_5_proba_classe_0
    ,vl_output_model_classificacao_7_5_proba_classe_1
    ,vl_output_model_classificacao_7_5_proba_classe_2
    ,vl_output_model_7_5
FROM big_union
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Cálculo das Métricas

In [25]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.metrics import precision_score, recall_score, f1_score
from metrics import PSC_A,PCC_A,PMC_A


# ==============================================================================
# CONFIGURAÇÕES
# ==============================================================================

psc_a_max_chuva = 1
pcc_a_erro = 1
pmc_a_erro = 5
pmc_a_min_chuva = 20

dict_classes_4 = {
    '4_1': {0: (0, 1), 1: (1, np.inf)},
    '4_2': {0: (0, 1), 1: (1, 51), 2: (51, np.inf)},
    '4_3': {0: (0, 1), 1: (1, 23), 2: (23, 51), 3: (51, np.inf)},
    '4_4': {0: (0, 1), 1: (1, 7), 2: (7, 51), 3: (51, np.inf)},
    '4_5': {0: (0, 1), 1: (1, 7), 2: (7, np.inf)}
}

dict_classes_5 = {
    '5_1': {0: (0, 1), 1: (1, np.inf)},
    '5_2': {0: (0, 1), 1: (1, 51), 2: (51, np.inf)},
    '5_3': {0: (0, 1), 1: (1, 23), 2: (23, 51), 3: (51, np.inf)},
    '5_4': {0: (0, 1), 1: (1, 7), 2: (7, 51), 3: (51, np.inf)},
    '5_5': {0: (0, 1), 1: (1, 7), 2: (7, np.inf)}
}

dict_classes_6 = {
    '6_1': {0: (0, 1), 1: (1, np.inf)},
    '6_2': {0: (0, 1), 1: (1, 51), 2: (51, np.inf)},
    '6_3': {0: (0, 1), 1: (1, 23), 2: (23, 51), 3: (51, np.inf)},
    '6_4': {0: (0, 1), 1: (1, 7), 2: (7, 51), 3: (51, np.inf)},
    '6_5': {0: (0, 1), 1: (1, 7), 2: (7, np.inf)}
}

dict_classes_7 = {
    '7_1': {0: (0, 1), 1: (1, np.inf)},
    '7_2': {0: (0, 1), 1: (1, 51), 2: (51, np.inf)},
    '7_3': {0: (0, 1), 1: (1, 23), 2: (23, 51), 3: (51, np.inf)},
    '7_4': {0: (0, 1), 1: (1, 7), 2: (7, 51), 3: (51, np.inf)},
    '7_5': {0: (0, 1), 1: (1, 7), 2: (7, np.inf)}
}


# ==============================================================================
# FUNÇÕES AUXILIARES
# ==============================================================================

def calcular_classe_por_threshold(vl_precipitacao: pd.Series, dict_classes: dict) -> pd.Series:
    """Deriva a classe real de precipitação a partir dos thresholds definidos."""
    return vl_precipitacao.apply(
        lambda x: next(
            (classe for classe, (lim_inf, lim_sup) in dict_classes.items()
             if lim_inf <= x < lim_sup),
            None
        )
    )


def calcular_metricas_regressao(
    y_true: pd.Series,
    y_pred: pd.Series,
    psc_a_max_chuva: float,
    pcc_a_erro: float,
    pmc_a_erro: float,
    pmc_a_min_chuva: float
) -> dict:
    """Calcula MAE, RMSE, R2 e métricas customizadas de precipitação para regressão."""
    return {
        'MAE':   mean_absolute_error(y_true, y_pred),
        'RMSE':  root_mean_squared_error(y_true, y_pred),
        'R2':    r2_score(y_true, y_pred),
        'PSC_A': PSC_A(y_true, y_pred, psc_a_max_chuva),
        'PCC_A': PCC_A(y_true, y_pred, pcc_a_erro),
        'PMC_A': PMC_A(y_true, y_pred, pmc_a_erro, pmc_a_min_chuva)
    }


def calcular_metricas_classificacao(
    y_true: pd.Series,
    y_pred: pd.Series,
    y_proba: pd.DataFrame
) -> dict:
    """Calcula precisão, recall, F1 e log-loss para classificação multiclasse."""
    classes = sorted(y_true.unique())
    return {
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall':    recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1':        f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }


# ==============================================================================
# MÉTRICAS DE REGRESSÃO — MODELOS 1, 2, 3
# ==============================================================================

modelos_regressao_simples = {
    '1_1': 'vl_output_model_1_1',
    '1_2': 'vl_output_model_1_2',
    '1_3': 'vl_output_model_1_3',
    '1_4': 'vl_output_model_1_4',
    '2_1': 'vl_output_model_2_1',
    '2_2': 'vl_output_model_2_2',
    '2_3': 'vl_output_model_2_3',
    '2_4': 'vl_output_model_2_4',
    '3_1': 'vl_output_model_3_1',
    '3_2': 'vl_output_model_3_2',
    '3_3': 'vl_output_model_3_3',
    '3_4': 'vl_output_model_3_4',
    '3_5': 'vl_output_model_3_5',
}

metricas_regressao = {}

y_true = df_big['vl_precipitacao']

for version, col_pred in modelos_regressao_simples.items():
    metricas_regressao[version] = calcular_metricas_regressao(
        y_true, df_big[col_pred],
        psc_a_max_chuva, pcc_a_erro, pmc_a_erro, pmc_a_min_chuva
    )

# Modelos 4, 5, 6, 7 — output final de regressão
for version in ['4_1', '4_2', '4_3', '4_4', '4_5',
                '5_1', '5_2', '5_3', '5_4', '5_5',
                '6_1', '6_2', '6_3', '6_4', '6_5',
                '7_1', '7_2', '7_3', '7_4', '7_5']:
    metricas_regressao[version] = calcular_metricas_regressao(
        y_true, df_big[f'vl_output_model_{version}'],
        psc_a_max_chuva, pcc_a_erro, pmc_a_erro, pmc_a_min_chuva
    )

df_metricas_regressao = pd.DataFrame(metricas_regressao).T
df_metricas_regressao.index.name = 'modelo'
print("=== Métricas de Regressão ===")
print(df_metricas_regressao.to_string())


# ==============================================================================
# MÉTRICAS DE CLASSIFICAÇÃO — MODELO 4
# ==============================================================================

metricas_classificacao = {}

for version, classes in dict_classes_4.items():
    y_true_classe = calcular_classe_por_threshold(y_true, classes)
    y_pred_classe = df_big[f'vl_output_model_classificacao_{version}'].astype(int)

    n_classes = len(classes)
    y_proba_dummy = pd.get_dummies(y_pred_classe).reindex(
        columns=sorted(classes.keys()), fill_value=0
    ).astype(float)

    metricas_classificacao[version] = calcular_metricas_classificacao(
        y_true_classe, y_pred_classe, y_proba_dummy
    )

# ==============================================================================
# MÉTRICAS DE CLASSIFICAÇÃO — MODELO 5 (argmax das probas)
# ==============================================================================

for version, classes in dict_classes_5.items():
    y_true_classe = calcular_classe_por_threshold(y_true, classes)

    cols_proba = [f'vl_output_model_classificacao_{version}_proba_classe_{c}'
                  for c in sorted(classes.keys())]
    y_proba = df_big[cols_proba]
    y_pred_classe = y_proba.values.argmax(axis=1)

    metricas_classificacao[version] = calcular_metricas_classificacao(
        y_true_classe, y_pred_classe, y_proba
    )

# ==============================================================================
# MÉTRICAS DE CLASSIFICAÇÃO — MODELO 6
# ==============================================================================

for version, classes in dict_classes_6.items():
    y_true_classe = calcular_classe_por_threshold(y_true, classes)
    y_pred_classe = df_big[f'vl_output_model_classificacao_{version}'].astype(int)

    y_proba_dummy = pd.get_dummies(y_pred_classe).reindex(
        columns=sorted(classes.keys()), fill_value=0
    ).astype(float)

    metricas_classificacao[version] = calcular_metricas_classificacao(
        y_true_classe, y_pred_classe, y_proba_dummy
    )

# ==============================================================================
# MÉTRICAS DE CLASSIFICAÇÃO — MODELO 7 (argmax das probas)
# ==============================================================================

for version, classes in dict_classes_7.items():
    y_true_classe = calcular_classe_por_threshold(y_true, classes)

    cols_proba = [f'vl_output_model_classificacao_{version}_proba_classe_{c}'
                  for c in sorted(classes.keys())]
    y_proba = df_big[cols_proba]
    y_pred_classe = y_proba.values.argmax(axis=1)

    metricas_classificacao[version] = calcular_metricas_classificacao(
        y_true_classe, y_pred_classe, y_proba
    )

df_metricas_classificacao = pd.DataFrame(metricas_classificacao).T
df_metricas_classificacao.index.name = 'modelo'
print("\n=== Métricas de Classificação ===")
print(df_metricas_classificacao.to_string())

=== Métricas de Regressão ===
             MAE       RMSE        R2      PSC_A      PCC_A      PMC_A
modelo                                                                
1_1     5.460297  10.150194  0.030744   6.428698  11.641076   0.473503
1_2     3.226563   7.859220  0.418903  75.401540  20.045932  12.336413
1_3     2.856736   7.221627  0.509364  77.500109  25.463811  18.179684
1_4     2.749362   7.092083  0.526808  80.507026  25.426790  18.507399
2_1     3.851167  10.705405 -0.078192  98.620269  27.436145   0.000000
2_2     3.435703  10.305428  0.000870  93.730913  27.348371   0.000000
2_3     3.387007  10.242454  0.013044  94.363519  28.181093   0.000000
2_4     3.371147  10.230124  0.015419  94.554669  28.244766   0.000000
3_1     3.391912  10.141943  0.032319  93.362567  27.925825   0.653885
3_2     3.409173  10.192159  0.022713  93.613490  27.957458   0.663461
3_3     3.427955  10.251761  0.011250  93.829567  27.960471   0.626396
3_4     3.392048  10.135542  0.033540  93.26004

In [26]:
sql("CREATE OR REPLACE TABLE tb_metricas_regressao AS (SELECT * FROM df_metricas_regressao)")
print("✅ tb_metricas_regressao salva com sucesso!")

sql("CREATE OR REPLACE TABLE tb_metricas_classificacao AS (SELECT * FROM df_metricas_classificacao)")
print("✅ tb_metricas_classificacao salva com sucesso!")

✅ tb_metricas_regressao salva com sucesso!
✅ tb_metricas_classificacao salva com sucesso!
